# 02 - Nettoyage et Feature Engineering

Objectif: produire un dataset propre, stable, et exploitable pour le scoring et la modelisation.

Sorties attendues:
- data/accidents_clean.csv (dataset nettoye)
- data/accidents_scored.csv (dataset enrichi avec anomaly_score)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)


In [ ]:
csv_params = {'sep': ',', 'encoding': 'latin1', 'low_memory': False}

df_caract = pd.read_csv('data/caracteristiques.csv', **csv_params)
df_lieux = pd.read_csv('data/lieux.csv', **csv_params)
df_usagers = pd.read_csv('data/usagers.csv', **csv_params)


In [ ]:
def normalize_num_acc(df):
    df = df.copy()
    df['num_acc'] = pd.to_numeric(df['num_acc'], errors='coerce')
    df = df.dropna(subset=['num_acc'])
    df['num_acc'] = df['num_acc'].astype('int64')
    return df

df_caract = normalize_num_acc(df_caract)
df_lieux = normalize_num_acc(df_lieux)
df_usagers = normalize_num_acc(df_usagers)

df_caract = df_caract.drop_duplicates()
df_lieux = df_lieux.drop_duplicates()
df_usagers = df_usagers.drop_duplicates()


In [ ]:
# Remplacement des codes -1 par NaN (numeriques)
for df in [df_caract, df_lieux, df_usagers]:
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].replace({-1: np.nan})


In [ ]:
# Nettoyage VMA
df_lieux['vma'] = pd.to_numeric(df_lieux['vma'], errors='coerce')
mask_gros = df_lieux['vma'] > 130
df_lieux.loc[mask_gros, 'vma'] = df_lieux.loc[mask_gros, 'vma'] / 10
df_lieux['vma'] = df_lieux['vma'].mask((df_lieux['vma'] > 130) | (df_lieux['vma'] <= 10), np.nan)
df_lieux['vma'] = df_lieux['vma'].fillna(df_lieux['vma'].median())


In [ ]:
# Nettoyage usagers
df_usagers = df_usagers.drop(columns=['id_vehicule'], errors='ignore')
df_usagers['catu'] = pd.to_numeric(df_usagers['catu'], errors='coerce')


In [ ]:
# Suppression des colonnes parasites
for df in [df_caract, df_lieux, df_usagers]:
    df.drop(columns=['Unnamed: 0'], errors='ignore', inplace=True)


In [ ]:
# Suppression des orphelins (num_acc absent de Caract)
ids_caract = set(df_caract['num_acc'])
df_lieux = df_lieux[df_lieux['num_acc'].isin(ids_caract)].copy()
df_usagers = df_usagers[df_usagers['num_acc'].isin(ids_caract)].copy()


In [ ]:
# Features usagers (par accident)
agg = df_usagers.groupby('num_acc').agg(
    nb_usagers=('num_acc', 'size'),
    nb_pietons=('catu', lambda s: (s == 3).sum())
).reset_index()
agg['pct_pietons'] = np.where(agg['nb_usagers'] > 0, agg['nb_pietons'] / agg['nb_usagers'], 0)


In [ ]:
# Fusion globale
df_acc = df_caract.merge(df_lieux, on='num_acc', how='inner')
df_acc = df_acc.merge(agg, on='num_acc', how='left')

# Harmonisation annee
if 'annee_x' in df_acc.columns:
    df_acc = df_acc.rename(columns={'annee_x': 'annee'})
if 'annee_y' in df_acc.columns:
    df_acc = df_acc.drop(columns=['annee_y'])

# Feature heure (hrmn -> heure)
df_acc['heure'] = df_acc['hrmn'].astype(str).str.split('.').str[0].str.zfill(4).str[:2]
df_acc['heure'] = pd.to_numeric(df_acc['heure'], errors='coerce')
df_acc['heure'] = df_acc['heure'].fillna(df_acc['heure'].median())

df_acc.to_csv('data/accidents_clean.csv', index=False)
print('accidents_clean.csv sauvegarde')


In [ ]:
# Isolation Forest pour un score d anomalie
cols_exclure = [
    'num_acc', 'hrmn', 'voie', 'com', 'dep', 'Unnamed: 0', 'annee'
]
X = df_acc.drop(columns=cols_exclure, errors='ignore')

preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

model_iso = IsolationForest(
    contamination='auto', n_estimators=200, random_state=42, n_jobs=-1
)

pipeline = Pipeline([('prep', preprocessor), ('model', model_iso)])
pipeline.fit(X)

scores = -pipeline.decision_function(X)
df_acc['anomaly_score'] = scores
df_acc['is_anomaly'] = (pipeline.predict(X) == -1).astype(int)

df_acc.to_csv('data/accidents_scored.csv', index=False)
print('accidents_scored.csv sauvegarde')
